In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasettfinal/fixed_cleaned_file.jsonl


In [ ]:
pip install trl

In [ ]:
pip install -U bitsandbytes

In [2]:
import os

# ✅ CRITICAL: Set single GPU BEFORE importing torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import torch
import gc
import numpy as np
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs visible: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("✅ Imports complete")

2025-12-16 17:18:19.672608: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765905499.693498     196 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765905499.699814     196 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

PyTorch version: 2.6.0+cu124
CUDA available: True
Number of GPUs visible: 1
GPU: Tesla P100-PCIE-16GB
✅ Imports complete


In [3]:
CONFIG = {
    "model_name": "mistralai/Mistral-7B-v0.1",
    "data_path": "/kaggle/input/datasettfinal/fixed_cleaned_file.jsonl",
    "output_dir": "/kaggle/working/mistral_finetuned",
    "max_seq_length": 512,
    
    # LoRA Configuration
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    
    # Training Configuration
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "max_steps": 500,
    "save_steps": 100,
    "logging_steps": 10,
}

print("✅ Configuration loaded")

✅ Configuration loaded


In [4]:
def check_environment():
    """Check and print environment information"""
    print("=" * 60)
    print("ENVIRONMENT CHECK")
    print("=" * 60)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"Number of GPUs: {torch.cuda.device_count()}")
        print(f"GPU 0: {torch.cuda.get_device_name(0)}")
        
        free_mem = torch.cuda.mem_get_info(0)[0] / 1024**3
        total_mem = torch.cuda.mem_get_info(0)[1] / 1024**3
        print(f"Memory: {free_mem:.1f} GB free / {total_mem:.1f} GB total")
    
    print("=" * 60)
    
    # Clear GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    print("✅ GPU memory cleared\n")

check_environment()

ENVIRONMENT CHECK
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
Number of GPUs: 1
GPU 0: Tesla P100-PCIE-16GB
Memory: 15.6 GB free / 15.9 GB total
✅ GPU memory cleared



In [5]:
def fix_jsonl_file(input_path, output_path):
    """Fix and validate JSONL file"""
    print(f"Fixing JSONL file: {input_path}")
    valid_lines = 0
    invalid_lines = 0
    
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line_num, line in enumerate(infile, 1):
            line = line.strip()
            if not line:
                continue
                
            try:
                data = json.loads(line)
                if "prompt" in data and "code" in data:
                    fixed_line = json.dumps(data, ensure_ascii=False)
                    outfile.write(fixed_line + "\n")
                    valid_lines += 1
                else:
                    invalid_lines += 1
            except json.JSONDecodeError:
                invalid_lines += 1
    
    print(f"Valid lines: {valid_lines}")
    print(f"Invalid lines: {invalid_lines}")
    print(f"Fixed file saved to: {output_path}\n")
    
    return output_path

print("✅ Function defined: fix_jsonl_file")

✅ Function defined: fix_jsonl_file


In [6]:
def load_and_prepare_dataset(data_path):
    """Load and prepare dataset for training"""
    print("Loading dataset...")
    dataset = load_dataset("json", data_files=data_path, split="train")
    
    if "code" in dataset.column_names:
        dataset = dataset.rename_column("code", "completion")
    
    def format_instruction(example):
        text = f"### Prompt:\n{example['prompt']}\n\n### Code:\n{example['completion']}"
        return {"text": text}
    
    dataset = dataset.map(format_instruction, remove_columns=dataset.column_names)
    
    print(f"Dataset size: {len(dataset)}")
    print(f"Sample entry:\n{dataset[0]['text'][:300]}...\n")
    
    return dataset

print("✅ Function defined: load_and_prepare_dataset")

✅ Function defined: load_and_prepare_dataset


In [7]:
def setup_model_and_tokenizer(model_name, max_seq_length):
    """
    Setup model and tokenizer with 4-bit quantization
    FIXED: Using float16 only (P100 compatible)
    """
    
    # Clear GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"Loading model: {model_name}")
    
    free_mem = torch.cuda.mem_get_info(0)[0] / 1024**3
    print(f"GPU 0 free memory: {free_mem:.2f} GB")
    
    # Create offload folder
    offload_folder = "/kaggle/working/offload"
    os.makedirs(offload_folder, exist_ok=True)
    
    # ✅ FIXED: Use float16 ONLY - P100 doesn't support bfloat16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,  # ✅ MUST be float16
        bnb_4bit_use_double_quant=True,
    )
    
    # Load tokenizer first
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, 
        trust_remote_code=True,
        use_fast=False
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    tokenizer.model_max_length = max_seq_length
    
    # Load model
    print("Loading model with 4-bit quantization (float16)...")
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map={"": 0},  # Force single GPU
        torch_dtype=torch.float16,  # ✅ MUST be float16
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    
    # Configure model
    model.config.use_cache = False
    model.config.pad_token_id = tokenizer.pad_token_id
    
    # Prepare for training
    model = prepare_model_for_kbit_training(model)
    
    # ✅ Verify no bfloat16 tensors
    print("\nVerifying tensor dtypes...")
    has_bf16 = False
    for name, param in model.named_parameters():
        if param.dtype == torch.bfloat16:
            has_bf16 = True
            print(f"⚠️ Found bfloat16: {name}")
            break
    
    if not has_bf16:
        print("✅ No bfloat16 tensors found - safe to train!")
    
    print(f"\n✅ Model loaded successfully!")
    print(f"✅ Tokenizer max length: {tokenizer.model_max_length}")
    
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    print(f"✅ GPU 0 memory used: {allocated:.2f} GB\n")
    
    return model, tokenizer

print("✅ Function defined: setup_model_and_tokenizer")

✅ Function defined: setup_model_and_tokenizer


In [8]:
def setup_lora(model, config):
    """Setup LoRA configuration"""
    print("Setting up LoRA...")
    
    lora_config = LoraConfig(
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=config["lora_target_modules"],
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    print()
    
    return model

print("✅ Function defined: setup_lora")

✅ Function defined: setup_lora


In [9]:
def setup_trainer(model, tokenizer, dataset, config):
    """
    Setup SFTTrainer
    FIXED: Disabled mixed precision to avoid AMP issues on P100
    """
    
    print("Setting up trainer...")
    
    from trl import SFTTrainer, SFTConfig
    import trl
    print(f"TRL version: {trl.__version__}")
    
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")
    
    # ✅ FIXED: Disable ALL mixed precision to avoid AMP/bfloat16 issues
    print("Using: NO mixed precision (fp16=False, bf16=False)")
    
    sft_config = SFTConfig(
        output_dir=config["output_dir"],
        per_device_train_batch_size=config["per_device_train_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        optim="paged_adamw_8bit",
        learning_rate=config["learning_rate"],
        lr_scheduler_type="cosine",
        warmup_steps=50,
        max_steps=config["max_steps"],
        save_steps=config["save_steps"],
        logging_steps=config["logging_steps"],
        
        # ✅ CRITICAL FIX: Disable ALL mixed precision
        fp16=False,
        bf16=False,
        
        gradient_checkpointing=True,
        save_total_limit=3,
        report_to="none",
        max_grad_norm=0.3,
        
        # Additional stability settings
        ddp_find_unused_parameters=False,
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        processing_class=tokenizer,
        args=sft_config,
    )
    
    print("✅ Trainer setup complete\n")
    return trainer

print("✅ Function defined: setup_trainer")

✅ Function defined: setup_trainer


In [10]:
def main():
    """Main training pipeline"""
    
    # Step 1: Check environment
    check_environment()
    
    # Step 2: Fix JSONL file
    fixed_data_path = "/kaggle/working/fixed_cleaned_file.jsonl"
    fix_jsonl_file(CONFIG["data_path"], fixed_data_path)
    
    # Step 3: Load dataset
    dataset = load_and_prepare_dataset(fixed_data_path)
    
    # Step 4: Setup model and tokenizer
    model, tokenizer = setup_model_and_tokenizer(
        CONFIG["model_name"],
        CONFIG["max_seq_length"]
    )
    
    # Step 5: Setup LoRA
    model = setup_lora(model, CONFIG)
    
    # Step 6: Setup trainer
    trainer = setup_trainer(model, tokenizer, dataset, CONFIG)
    
    # Step 7: Train
    print("=" * 60)
    print("STARTING TRAINING")
    print("=" * 60)
    
    trainer.train()
    
    # Step 8: Save model
    print("\n✅ Saving final model...")
    trainer.save_model(CONFIG["output_dir"])
    tokenizer.save_pretrained(CONFIG["output_dir"])
    
    print(f"\n🎉 Training complete! Model saved to: {CONFIG['output_dir']}")

print("✅ Function defined: main")

✅ Function defined: main


In [ ]:
# Clear GPU before starting
gc.collect()
torch.cuda.empty_cache()

# Run training
if __name__ == "__main__":
    main()

ENVIRONMENT CHECK
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
Number of GPUs: 1
GPU 0: Tesla P100-PCIE-16GB
Memory: 15.6 GB free / 15.9 GB total
✅ GPU memory cleared

Fixing JSONL file: /kaggle/input/datasettfinal/fixed_cleaned_file.jsonl
Valid lines: 399
Invalid lines: 0
Fixed file saved to: /kaggle/working/fixed_cleaned_file.jsonl

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/399 [00:00<?, ? examples/s]

Dataset size: 399
Sample entry:
### Prompt:
Draw a circle with radius 3.5 cm.

### Code:
from manim import *

class DrawCircle(Scene):
    def construct(self):
        radius = 1.37  # 3.5 cm ≈ 1.37 units (1 unit ≈ 2.54 cm)
        circle = Circle(radius=radius, color=BLUE)
        self.play(Create(circle))
        self.wait(2)
...

Loading model: mistralai/Mistral-7B-v0.1
GPU 0 free memory: 15.64 GB
Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


...

Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/399 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


✅ Trainer setup complete

STARTING TRAINING


Step,Training Loss
10,0.665800
20,0.545900
